# SinhalaMMLU: zero-shot baseline for Qwen3.5-4B

Scores the untuned model the way the official SinhalaMMLU evaluation does: the same Sinhala prompt, and the prediction is the option digit with the highest next-token logit. Nothing is sampled, so the run is deterministic and every question gets an answer.

**Before you run**
1. *Runtime → Change runtime type → GPU.* A T4 works (about 9 GB used); an L4 or A100 is faster.
2. Accept the dataset terms at [hf.co/datasets/naist-nlp/SinhalaMMLU](https://huggingface.co/datasets/naist-nlp/SinhalaMMLU), then add your Hugging Face token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar).

**What the number means.** The public Hugging Face split is Easy only (14 subjects), so it measures AccE, a third of the ranking metric. For the macro score, point `DATA_GLOB` at the organisers' dev set.

In [ ]:
# Qwen3.5 needs transformers 5.x; pinned so reruns give the same numbers.
%pip install -q "transformers==5.19.0" accelerate
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader

## Settings
`LIMIT = 20` runs a quick smoke test; `0` runs every question. For the dev set, mount Drive in the next cell and set `DATA_GLOB` to its files, for example `/content/drive/MyDrive/sinhalammlu_dev/**/*.json`. Both `.json` (a list of questions) and `.jsonl` files are read.

In [ ]:
MODEL_ID = "Qwen/Qwen3.5-4B"  # @param {type:"string"}
DATA_GLOB = "/content/SinhalaMMLU/TEST/**/*.json"  # @param {type:"string"}
OUT_PATH = "/content/predictions.jsonl"  # @param {type:"string"}
LIMIT = 0  # @param {type:"integer"}

In [ ]:
from huggingface_hub import snapshot_download

# The public split is gated: the token comes from the HF_TOKEN Colab secret.
if DATA_GLOB.startswith("/content/SinhalaMMLU/"):
    snapshot_download("naist-nlp/SinhalaMMLU", repo_type="dataset", local_dir="/content/SinhalaMMLU")
# Dev set on Google Drive instead:
# from google.colab import drive; drive.mount("/content/drive")

## Evaluation code
Same logic as `eval_baseline.py`. The prompt strings are copied verbatim from the official `src/util_prompt.py`.

In [ ]:
import glob
import json
import sys
import zipfile
from collections import Counter

import pandas as pd
import torch
import transformers
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

SYSTEM = "You must output the Answer"
INTRO = "මෙය {} විෂයයට අදාළ බහුවරණ ප්‍රශ්නයකි.\n"
INSTRUCTION = {
    "easy": "පහත ප්‍රශ්නයට 1, 2, 3, 4 යන පිළිතුරුවලින් නිවැරදි හෝ ඉතාමත් ගැළපෙන පිළිතුර තෝරන්න.\n\n",
    "medium": "පහත ප්‍රශ්නයට 1, 2, 3, 4  යන පිළිතුරුවලින් නිවැරදි හෝ ඉතාමත් ගැළපෙන පිළිතුර තෝරාගන්න.\n\n",
    "hard": "පහත ප්‍රශ්නයට 1, 2, 3, 4, 5 යන පිළිතුරුවලින් නිවැරදි හෝ ඉතාමත් ගැළපෙන පිළිතුර තෝරාගන්න.\n\n",
}


def load_items(pattern: str) -> list[dict]:
    """Read every .json (a list of questions) or .jsonl file that matches the glob."""
    items = []
    for path in sorted(glob.glob(pattern, recursive=True)):
        with open(path, encoding="utf-8") as f:
            items += [json.loads(line) for line in f if line.strip()] if path.endswith(".jsonl") else json.load(f)
    if not items:
        raise FileNotFoundError(f"No questions found for {pattern!r}")
    return items


def level_of(item: dict) -> str:
    return (item.get("level") or item["metadata"]["difficulty"]).lower()


def build_prompt(item: dict) -> str:
    """Official zero-shot prompt: subject intro, level instruction, question, numbered options."""
    subject = item.get("metadata", {}).get("subject_original") or item["subject"]
    options = "\n".join(f"{i}. {choice.strip()}" for i, choice in enumerate(item["choices"], start=1))
    return f"{INTRO.format(subject)}{INSTRUCTION[level_of(item)]}ප්‍රශ්නය: {item['question']}\n{options}\nපිළිතුර:"


def messages(item: dict) -> list[dict]:
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": build_prompt(item)}]


@torch.inference_mode()
def predict(model, tokenizer, item: dict, digit_ids: list[int]) -> int:
    """Return the 1-based option whose digit token has the highest next-token logit."""
    inputs = tokenizer.apply_chat_template(
        messages(item),
        add_generation_prompt=True,
        enable_thinking=False,  # Qwen3.5 thinks by default; then the next token is reasoning, not a digit
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    logits = model(**inputs, logits_to_keep=1).logits[0, -1, digit_ids[: len(item["choices"])]]
    assert torch.isfinite(logits).all(), f"non-finite logits for q_no={item.get('q_no')}"
    return int(logits.argmax()) + 1

## Load the model
The first run downloads about 9 GB of weights.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# Resolves to the text-only Qwen3_5ForCausalLM: the vision tower and MTP head are not loaded.
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="cuda").eval()

digit_tokens = [tokenizer.encode(str(d), add_special_tokens=False) for d in range(1, 6)]
assert all(len(t) == 1 for t in digit_tokens), "each option digit must be a single token"
digit_ids = [t[0] for t in digit_tokens]
n_params = sum(p.numel() for p in model.parameters())
print(f"{type(model).__name__}: {n_params / 1e9:.3f}B parameters loaded")

## Check the input
Prints exactly what the model sees for the first question. It should end with an empty `<think>` block.

In [ ]:
items = load_items(DATA_GLOB)[: LIMIT or None]
print(f"{len(items)} questions by level:", dict(Counter(level_of(item) for item in items)))
print(tokenizer.apply_chat_template(messages(items[0]), add_generation_prompt=True, enable_thinking=False, tokenize=False))

## Run

In [ ]:
preds = [predict(model, tokenizer, item, digit_ids) for item in tqdm(items)]

## Score
Same comparison as the official `get_accuracy_all.py`: both sides converted to numbers first. Rows without a gold answer (the hidden test set) are left out.

In [ ]:
df = pd.DataFrame({
    "level": [level_of(item) for item in items],
    "category": [item.get("category") for item in items],
    "gold": pd.to_numeric(pd.Series([item.get("answer") for item in items]), errors="coerce"),
    "pred": preds,
})
scored = df.dropna(subset=["gold"]).assign(correct=lambda d: d["gold"] == d["pred"])

if scored.empty:
    macro = None
    print("No gold answers in this data: predictions only.")
else:
    by_level = scored.groupby("level")["correct"].agg(accuracy="mean", n="size").reindex(["easy", "medium", "hard"]).dropna()
    macro = by_level["accuracy"].mean()
    display(by_level.style.format({"accuracy": "{:.2%}", "n": "{:.0f}"}))
    print(f"Macro (ranking metric): {macro:.2%} over {len(by_level)} of 3 levels | overall: {scored['correct'].mean():.2%}")
    display(scored.groupby("category")["correct"].agg(accuracy="mean", n="size").sort_values("accuracy").style.format({"accuracy": "{:.2%}"}))

## Save
Writes `predictions.jsonl`, zips it as `predictions.zip` for Codabench, and records the run settings in `run_info.json`. Colab files are lost when the runtime ends, so download them or copy them to Drive.

In [ ]:
with open(OUT_PATH, "w", encoding="utf-8") as f:
    for item, pred in zip(items, preds):
        row = {key: item.get(key) for key in ("q_no", "subject", "category")}
        f.write(json.dumps(row | {"level": level_of(item), "answer": str(pred)}, ensure_ascii=False) + "\n")

zip_path = OUT_PATH.rsplit("/", 1)[0] + "/predictions.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUT_PATH, arcname="predictions.jsonl")

run_info = {
    "model": MODEL_ID, "params": n_params, "dtype": "bfloat16", "data": DATA_GLOB, "questions": len(items),
    "macro": macro, "transformers": transformers.__version__, "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name() if torch.cuda.is_available() else "cpu",
}
with open(OUT_PATH.rsplit("/", 1)[0] + "/run_info.json", "w") as f:
    json.dump(run_info, f, indent=2)
print(json.dumps(run_info, indent=2))

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(zip_path)